In [85]:
#Scripts
import src.graphon as graphon
import src.metrics as metrics
from torch_geometric.utils import degree

#Libraries
import torch
from torch_geometric.data import Data

#Autoreload module
"""
These magic lines allow to automatically reload any module.py file when there are some changes on it, without the needing of restart the kernel anytime a new change is applied to the module script
"""
%load_ext autoreload
%autoreload 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [86]:
#Generate some random W-graphs
modes = ["ring", "sbm", "random"] #Types of graphs
n_graphs = 5 #Number of graphs to generate for each mode
device = "cpu"
n = 50 #Number of nodes in each graph
K = 5 #Discrete state space
graph_set = {mode : None for mode in modes}
for mode in graph_set.keys():
    graphs = [None] * n_graphs
    z = torch.rand(n, device = device) #Latent positions
    z = torch.sort(z).values #Node 0 has the smallest latent position

    # Edges (undirected)
    start, end = torch.triu_indices(n, n, offset = 1, device = device) #All possible combinations of undirected links
    z_start = z[start] #Latent positions of source nodes
    z_end = z[end] #Latent positions of ending nodes

    for i in range(n_graphs):
        if mode == "ring":
            probs = graphon.ring_graphon(z_start, z_end)
        elif mode == "sbm":
            blocks = torch.tensor([[0.8, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]], device = "cpu")
            probs = graphon.sbm_graphon(z_start, z_end, blocks)
        elif mode == "random":
            p = 0.5
            probs = torch.full_like(z_start, p, device = "cpu")
    
        # Sample links with inverse transform
        u = torch.rand_like(probs) #Uniform distributions over [0, 1]
        links = u < probs #Bernoulli samples

        # Keep only existing edges
        edge_start = start[links]
        edge_end = end[links]

        # Create edge set
        edges = torch.cat(
            [
                torch.stack([edge_start, edge_end]), #Connection i->j
                torch.stack([edge_end, edge_start]), #Connection j->i
            ],
            dim = 1
        )

        # Sample node features
        x = torch.randint(low = 0, high = K, size = (n, 1), device = device)

        # Create Data pytorch object
        graph = Data(
            x = x,
            edge_index = edges,
            num_nodes = n,
        )

        graphs[i] = graph

    graph_set[mode] = graphs

In [87]:
#Density and clustering coefficients
for mode in graph_set.keys():
    print(f"{mode} graphs:")
    densities = [metrics.density(graph) for graph in graph_set[mode]]
    loc_clust = [metrics.local_clustering_coeff(graph) for graph in graph_set[mode]]
    global_clust = [metrics.global_clustering_coeff(graph) for graph in graph_set[mode]]
    print(f"Density: {densities}")
    #print(f"Local: {loc_clust}")
    print(f"Global clustering: {global_clust}")
    print("="*50)

ring graphs:
Density: [0.1853061224489796, 0.18775510204081633, 0.19346938775510203, 0.18857142857142858, 0.1820408163265306]
Global clustering: [tensor(0.5127), tensor(0.5331), tensor(0.5392), tensor(0.5635), tensor(0.5426)]
sbm graphs:
Density: [0.3363265306122449, 0.3289795918367347, 0.32816326530612244, 0.31755102040816324, 0.33224489795918366]
Global clustering: [tensor(0.5465), tensor(0.5920), tensor(0.5915), tensor(0.5636), tensor(0.5611)]
random graphs:
Density: [0.4987755102040816, 0.4946938775510204, 0.5004081632653061, 0.47183673469387755, 0.4963265306122449]
Global clustering: [tensor(0.4906), tensor(0.4853), tensor(0.4929), tensor(0.4792), tensor(0.4995)]


In [88]:
#Centrality coefficients
for mode in graph_set.keys():
    print(f"{mode} graphs:")
    harmonic_cent = [metrics.harmonic_centrality(graph, centralization = "freeman") for graph in graph_set[mode]]
    betweenness_cent = [metrics.betweenness_centrality(graph, centralization = "freeman") for graph in graph_set[mode]]
    pagerank_cent = [metrics.pagerank_centrality(graph, centralization = "entropy") for graph in graph_set[mode]]
    print(f"Harmonic: {harmonic_cent}")
    print(f"Betweenness: {betweenness_cent}")
    print(f"Pagerank: {pagerank_cent}")
    print("="*50)

ring graphs:
Harmonic: [tensor(0.1675), tensor(0.1537), tensor(0.2311), tensor(0.2092), tensor(0.2315)]
Betweenness: [tensor(0.1035), tensor(0.0889), tensor(0.1051), tensor(0.1091), tensor(0.1115)]
Pagerank: [tensor(0.0036), tensor(0.0041), tensor(0.0034), tensor(0.0039), tensor(0.0047)]
sbm graphs:
Harmonic: [tensor(0.1508), tensor(0.1207), tensor(0.1250), tensor(0.1550), tensor(0.1599)]
Betweenness: [tensor(0.0186), tensor(0.0135), tensor(0.0260), tensor(0.0229), tensor(0.0240)]
Pagerank: [tensor(0.0028), tensor(0.0017), tensor(0.0022), tensor(0.0036), tensor(0.0025)]
random graphs:
Harmonic: [tensor(0.1182), tensor(0.1437), tensor(0.0952), tensor(0.1675), tensor(0.1633)]
Betweenness: [tensor(0.0066), tensor(0.0076), tensor(0.0059), tensor(0.0086), tensor(0.0081)]
Pagerank: [tensor(0.0014), tensor(0.0016), tensor(0.0013), tensor(0.0020), tensor(0.0022)]


In [89]:
#Homophily coefficients
for mode in graph_set.keys():
    print(f"{mode} graphs:")
    jsd_informativity = [metrics.jsd_informativity(graph) for graph in graph_set[mode]]
    adj_homophily = [metrics.adjusted_homophily(graph) for graph in graph_set[mode]]
    print(f"JSD: {jsd_informativity}")
    print(f"Adjusted: {adj_homophily}")
    print("="*50)

ring graphs:
JSD: [tensor(0.0048), tensor(0.0064), tensor(0.0034), tensor(0.0098), tensor(0.0041)]
Adjusted: [tensor(-0.0039), tensor(0.0003), tensor(-0.0171), tensor(0.0118), tensor(-0.0214)]
sbm graphs:
JSD: [tensor(0.0014), tensor(0.0038), tensor(0.0028), tensor(0.0039), tensor(0.0018)]
Adjusted: [tensor(-0.0334), tensor(-0.0259), tensor(-0.0481), tensor(-0.0162), tensor(-0.0352)]
random graphs:
JSD: [tensor(0.0021), tensor(0.0031), tensor(0.0029), tensor(0.0022), tensor(0.0010)]
Adjusted: [tensor(-0.0232), tensor(-0.0167), tensor(-0.0073), tensor(-0.0199), tensor(-0.0166)]
